# SmartRetail Insight Studio
## Interactive Sales Intelligence & API-Enriched Data Quality Analyzer

**Project type:** End-to-end Python data application  
**Development format:** Jupyter Notebook  
**Final output:** Browser-based interactive web interface  
**Core technologies:** Python, Pandas, Requests, JSON, Matplotlib, Gradio

### Project objective

This project combines the supplied hands-on concepts into one professional application:

- CSV ingestion
- data-quality inspection
- duplicate and invalid-record handling
- missing-value treatment
- category standardisation
- public API GET requests
- JSON-to-DataFrame conversion
- API failure fallback
- table merging
- revenue calculation
- category analytics
- business-rule insights
- reusable functions
- CSV report generation
- interactive web UI

**Workflow:** Raw Sales → Quality Check → Cleaning → API Enrichment → Merge → Analytics → Reports → Web Dashboard

## 1. Problem Statement

Retail organisations frequently maintain transaction data locally while product
attributes such as brand, rating and stock are obtained from an external service.

This application creates a single analysis pipeline that cleans the local data,
retrieves product information, handles API failure safely, combines the sources,
calculates business metrics and presents the results through a browser interface.

The notebook remains the main development and documentation environment, while the
final Gradio section acts as the user-facing application.

## 2. Install and Verify Dependencies

**Pandas** is used for tabular processing. **Requests** handles HTTP GET requests.
**Matplotlib** produces the revenue chart. **Gradio** converts the Python workflow
into an interactive web application.

In [ ]:
%pip install -q pandas requests matplotlib gradio

## 3. Configure the Project Workspace

The project is organised into separate data, module and output directories.
This follows the project-structure concept from the supplied Python module exercise
and keeps generated reports separate from input data.

In [ ]:
from pathlib import Path
import pandas as pd
import requests
import matplotlib.pyplot as plt
import gradio as gr

PROJECT = Path("smart_retail_insight_studio")
DATA_DIR = PROJECT / "data"
MODULE_DIR = PROJECT / "modules"
OUTPUT_DIR = PROJECT / "output"

for folder in (DATA_DIR, MODULE_DIR, OUTPUT_DIR):
    folder.mkdir(parents=True, exist_ok=True)

API_URL = "https://dummyjson.com/products"
API_LIMIT = 10

print("Project:", PROJECT.resolve())
print("API:", API_URL)

## 4. Create the Local Sales Dataset

The supplied sales dataset intentionally contains common quality problems:
a duplicate order, inconsistent category formatting, one missing price and
one invalid negative quantity.

Creating the dataset inside the notebook makes the project reproducible.

In [ ]:
sales_csv = (
    "order_id,order_date,product_id,product_name,category,price,quantity,payment_status\n"
    "O101,2026-08-01,1,Mascara, beauty ,799,2,Paid\n"
    "O102,2026-08-02,2,Eye Shadow,BEAUTY,1599,1,Paid\n"
    "O103,2026-08-03,3,Face Powder,Beauty,1199,3,Paid\n"
    "O104,2026-08-04,4,Lipstick,beauty,999,2,Pending\n"
    "O104,2026-08-04,4,Lipstick,beauty,999,2,Pending\n"
    "O105,2026-08-05,5,Nail Polish,BEAUTY,699,4,Paid\n"
    "O106,2026-08-06,6,CK One Perfume,fragrance,3999,1,Paid\n"
    "O107,2026-08-07,7,Coco Noir,Fragrances,,1,Paid\n"
    "O108,2026-08-08,8,Dior Jadore,FRAGRANCES,7499,1,Paid\n"
    "O109,2026-08-09,9,Dolce Shine,fragrance,5999,-1,Paid\n"
    "O110,2026-08-10,10,Gucci Bloom,Fragrances,6499,2,Failed\n"
    "O111,2026-08-11,1,Mascara,Beauty,849,2,Paid\n"
)

sales_path = DATA_DIR / "sales_data.csv"
sales_path.write_text(sales_csv, encoding="utf-8")
print("Created:", sales_path)

## 5. Create the API Fallback Dataset

The API exercise specifies a fallback CSV for situations in which the public API
cannot be reached. The fallback contains the product fields required by the
analysis pipeline.

The application records the source as either **Live API** or **Fallback CSV**.

In [ ]:
fallback_csv = (
    "product_id,title,brand,category,api_price,rating,stock\n"
    "1,Essence Mascara Lash Princess,Essence,beauty,9.99,4.94,5\n"
    "2,Eyeshadow Palette with Mirror,Glamour Beauty,beauty,19.99,4.28,44\n"
    "3,Powder Canister,Velvet Touch,beauty,14.99,3.82,59\n"
    "4,Red Lipstick,Chic Cosmetics,beauty,12.99,4.51,68\n"
    "5,Red Nail Polish,Nail Couture,beauty,8.99,3.91,71\n"
    "6,Calvin Klein CK One,Calvin Klein,fragrances,49.99,4.85,17\n"
    "7,Chanel Coco Noir Eau De,Chanel,fragrances,129.99,4.26,41\n"
    "8,Dior Jadore,Dior,fragrances,89.99,4.31,91\n"
    "9,Dolce Shine Eau de,Dolce and Gabbana,fragrances,69.99,3.77,3\n"
    "10,Gucci Bloom Eau de,Gucci,fragrances,79.99,4.69,93\n"
)

fallback_path = DATA_DIR / "products_fallback.csv"
fallback_path.write_text(fallback_csv, encoding="utf-8")
print("Created:", fallback_path)

## 6. Load and Inspect Raw Data

Before modifying the dataset, the application records the original number of
rows, missing values and duplicate order IDs. This establishes a measurable
data-quality baseline.

In [ ]:
sales = pd.read_csv(sales_path)

raw_rows = len(sales)
raw_columns = len(sales.columns)
raw_missing = int(sales.isna().sum().sum())
raw_duplicates = int(sales["order_id"].duplicated().sum())

print("Rows:", raw_rows)
print("Columns:", raw_columns)
print("Missing values:", raw_missing)
print("Duplicate order IDs:", raw_duplicates)

display(sales)

## 7. Clean and Standardise the Sales Data

The cleaning rules directly follow the supplied data-cleaning exercise:

1. Remove duplicate order IDs.
2. Strip unnecessary spaces.
3. Standardise category names.
4. Convert `Fragrance` to `Fragrances`.
5. Fill missing price values using the median price.
6. Remove rows where quantity is not greater than zero.

The cleaning report makes each transformation measurable.

In [ ]:
clean_sales = sales.drop_duplicates(subset="order_id").copy()

clean_sales["category"] = (
    clean_sales["category"]
    .astype(str)
    .str.strip()
    .str.title()
    .replace({"Fragrance": "Fragrances"})
)

missing_prices_filled = int(clean_sales["price"].isna().sum())
median_price = clean_sales["price"].median()
clean_sales["price"] = clean_sales["price"].fillna(median_price)

invalid_quantity_rows = int((clean_sales["quantity"] <= 0).sum())
clean_sales = clean_sales[clean_sales["quantity"] > 0].copy()

clean_sales["revenue"] = clean_sales["price"] * clean_sales["quantity"]

cleaning_report = pd.DataFrame({
    "Metric": [
        "Original rows",
        "Duplicate rows removed",
        "Missing prices filled",
        "Invalid quantity rows removed",
        "Clean rows"
    ],
    "Value": [
        raw_rows,
        raw_duplicates,
        missing_prices_filled,
        invalid_quantity_rows,
        len(clean_sales)
    ]
})

display(cleaning_report)
display(clean_sales)

## 8. Retrieve Product Data from the Public API

The public-API exercise demonstrates the GET request, parameters and JSON response.
This implementation requests the required products, validates the HTTP response,
converts JSON records into a DataFrame and selects the business fields.

If the API request fails, the fallback CSV is loaded automatically.

In [ ]:
def fetch_products():
    try:
        response = requests.get(
            API_URL,
            params={"limit": API_LIMIT},
            timeout=10
        )
        response.raise_for_status()

        payload = response.json()
        products = pd.DataFrame(payload["products"])

        products = products[
            ["id", "title", "brand", "category", "price", "rating", "stock"]
        ].rename(columns={
            "id": "product_id",
            "category": "api_category",
            "price": "api_price"
        })

        return products, "Live API", None

    except (requests.RequestException, ValueError, KeyError) as error:
        products = pd.read_csv(fallback_path)
        products = products.rename(columns={"category": "api_category"})
        return products, "Fallback CSV", type(error).__name__

products, data_source, api_error = fetch_products()

print("Product data source:", data_source)
print("API status:", api_error or "Successful")
display(products)

## 9. Merge Sales and Product Information

`product_id` is the common key between the two tables.

A left merge preserves the cleaned sales records while enriching them with brand,
API price, rating and stock information.

In [ ]:
final_data = clean_sales.merge(
    products[
        [
            "product_id",
            "title",
            "brand",
            "api_category",
            "api_price",
            "rating",
            "stock"
        ]
    ],
    on="product_id",
    how="left"
)

display(final_data)

## 10. Calculate Revenue and Inventory Status

The local sale price remains the primary value. If it is unavailable, the API
price is used.

Revenue is calculated as **price × quantity**. Stock below 10 units is classified
as **Low Stock**. Revenue of at least ₹5,000 is classified as **High Value**, using
the business rule introduced in the supplied cleaning exercise.

In [ ]:
final_data["price"] = final_data["price"].fillna(final_data["api_price"])
final_data["revenue"] = final_data["price"] * final_data["quantity"]

final_data["stock_status"] = "Available"
final_data.loc[final_data["stock"] < 10, "stock_status"] = "Low Stock"

final_data["order_type"] = "Normal"
final_data.loc[final_data["revenue"] >= 5000, "order_type"] = "High Value"

display(
    final_data[
        [
            "order_id", "product_name", "brand", "price",
            "quantity", "revenue", "rating",
            "stock_status", "order_type"
        ]
    ]
)

## 11. Generate Category-Level Business Analytics

The supplied main project aggregates orders, quantity, revenue and average rating
by category. This project retains that structure and exposes it as a dashboard
dataset.

In [ ]:
category_summary = (
    final_data.groupby("category")
    .agg(
        total_orders=("order_id", "count"),
        total_quantity=("quantity", "sum"),
        total_revenue=("revenue", "sum"),
        average_rating=("rating", "mean")
    )
    .reset_index()
)

category_summary["total_revenue"] = category_summary["total_revenue"].round(2)
category_summary["average_rating"] = category_summary["average_rating"].round(2)

display(category_summary)

## 12. Add Reusable Business Queries and Rule-Based Insights

The supplied exercises ask for paid orders, low-stock products and top products
by revenue. These are implemented as reusable functions.

The insight engine is deliberately **rule-based**, not machine learning. It
translates measured values into concise statements for the dashboard.

In [ ]:
def get_paid_orders(data):
    return data[data["payment_status"].eq("Paid")].copy()

def get_low_stock(data):
    return data[data["stock"] < 10][
        ["product_name", "brand", "stock", "stock_status"]
    ].drop_duplicates()

def get_top_products(data, n=3):
    return (
        data.groupby(["product_id", "product_name"], as_index=False)["revenue"]
        .sum()
        .sort_values("revenue", ascending=False)
        .head(n)
    )

def generate_insights(data, summary, source):
    top_category = summary.sort_values(
        "total_revenue", ascending=False
    ).iloc[0]["category"]

    return [
        f"Total calculated revenue: ₹{data['revenue'].sum():,.2f}",
        f"Paid transactions retained: {(data['payment_status'] == 'Paid').sum()}",
        f"High Value transactions: {(data['order_type'] == 'High Value').sum()}",
        f"Low Stock records: {(data['stock'] < 10).sum()}",
        f"Highest-revenue category: {top_category}",
        f"Product enrichment source: {source}"
    ]

for message in generate_insights(final_data, category_summary, data_source):
    print("•", message)

print("\nTop products")
display(get_top_products(final_data))

print("\nLow stock")
display(get_low_stock(final_data))

## 13. Create the Revenue Chart

The main project creates a revenue-by-category chart and saves it as an image.
This implementation wraps that logic inside a function so the same chart can be
displayed both in the notebook and in the web interface.

In [ ]:
def create_revenue_chart(summary):
    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.bar(summary["category"], summary["total_revenue"])
    ax.set_title("Revenue by Category")
    ax.set_xlabel("Category")
    ax.set_ylabel("Revenue (₹)")
    fig.tight_layout()
    return fig

chart = create_revenue_chart(category_summary)
display(chart)
chart.savefig(OUTPUT_DIR / "revenue_chart.png", dpi=150, bbox_inches="tight")
plt.close(chart)

print("Created:", OUTPUT_DIR / "revenue_chart.png")

## 14. Save the Analytical Outputs

The project produces reusable CSV reports rather than leaving results only in
the notebook output.

This includes the enriched sales report, category summary, API product data,
paid orders, low-stock products and top-revenue products.

In [ ]:
final_data.to_csv(OUTPUT_DIR / "final_sales_report.csv", index=False)
category_summary.to_csv(OUTPUT_DIR / "final_category_summary.csv", index=False)
products.to_csv(OUTPUT_DIR / "api_products.csv", index=False)
get_paid_orders(final_data).to_csv(OUTPUT_DIR / "paid_orders.csv", index=False)
get_low_stock(final_data).to_csv(OUTPUT_DIR / "low_stock_products.csv", index=False)
get_top_products(final_data).to_csv(OUTPUT_DIR / "top_products.csv", index=False)

print("Generated files:")
for path in sorted(OUTPUT_DIR.iterdir()):
    print(" -", path.name)

## 15. Build One Reusable End-to-End Pipeline

The Python project-structure exercise demonstrates separating reusable functions
from notebook code. The following pipeline function packages the complete analysis
into one callable operation.

This is important for the final web interface because the UI should call the
pipeline instead of depending on previously executed notebook cells.

In [ ]:
def run_pipeline(sales_file=None):
    source = Path(sales_file) if sales_file else sales_path
    raw = pd.read_csv(source)

    report = {
        "raw_rows": len(raw),
        "raw_missing": int(raw.isna().sum().sum()),
        "duplicates_removed": int(raw["order_id"].duplicated().sum())
    }

    data = raw.drop_duplicates(subset="order_id").copy()

    data["category"] = (
        data["category"]
        .astype(str)
        .str.strip()
        .str.title()
        .replace({"Fragrance": "Fragrances"})
    )

    report["missing_prices_filled"] = int(data["price"].isna().sum())
    data["price"] = data["price"].fillna(data["price"].median())

    report["invalid_quantity_rows_removed"] = int((data["quantity"] <= 0).sum())
    data = data[data["quantity"] > 0].copy()

    data["revenue"] = data["price"] * data["quantity"]

    api_products, source_name, api_error_name = fetch_products()

    data = data.merge(
        api_products[
            [
                "product_id", "title", "brand", "api_category",
                "api_price", "rating", "stock"
            ]
        ],
        on="product_id",
        how="left"
    )

    data["price"] = data["price"].fillna(data["api_price"])
    data["revenue"] = data["price"] * data["quantity"]

    data["stock_status"] = "Available"
    data.loc[data["stock"] < 10, "stock_status"] = "Low Stock"

    data["order_type"] = "Normal"
    data.loc[data["revenue"] >= 5000, "order_type"] = "High Value"

    summary = (
        data.groupby("category")
        .agg(
            total_orders=("order_id", "count"),
            total_quantity=("quantity", "sum"),
            total_revenue=("revenue", "sum"),
            average_rating=("rating", "mean")
        )
        .reset_index()
    )

    summary["total_revenue"] = summary["total_revenue"].round(2)
    summary["average_rating"] = summary["average_rating"].round(2)

    report.update({
        "clean_rows": len(data),
        "total_revenue": float(data["revenue"].sum()),
        "paid_orders": int((data["payment_status"] == "Paid").sum()),
        "high_value_orders": int((data["order_type"] == "High Value").sum()),
        "low_stock_records": int((data["stock"] < 10).sum()),
        "data_source": source_name,
        "api_status": api_error_name or "Successful"
    })

    return data, summary, report

## 16. Design the Final Web Interface

The interface is designed as a compact analytics dashboard rather than a plain
data table.

It contains:

- CSV upload
- KPI cards
- revenue visualisation
- generated insights
- cleaned/enriched data table
- category summary
- top products
- low-stock products
- data-quality information
- downloadable CSV reports

This gives the project a practical software-application layer while retaining
the notebook as the documented development environment.

In [ ]:
def analyze_for_ui(uploaded_file):
    try:
        data, summary, report = run_pipeline(uploaded_file)

        clean_file = OUTPUT_DIR / "ui_final_sales_report.csv"
        summary_file = OUTPUT_DIR / "ui_category_summary.csv"

        data.to_csv(clean_file, index=False)
        summary.to_csv(summary_file, index=False)

        kpis = f'''
        <div style="display:grid;grid-template-columns:repeat(4,1fr);gap:12px;">
          <div style="padding:18px;border-radius:16px;background:#111827;color:white;">
            <small>TOTAL REVENUE</small><h2>₹{report["total_revenue"]:,.2f}</h2>
          </div>
          <div style="padding:18px;border-radius:16px;background:#111827;color:white;">
            <small>CLEAN RECORDS</small><h2>{report["clean_rows"]}</h2>
          </div>
          <div style="padding:18px;border-radius:16px;background:#111827;color:white;">
            <small>HIGH VALUE</small><h2>{report["high_value_orders"]}</h2>
          </div>
          <div style="padding:18px;border-radius:16px;background:#111827;color:white;">
            <small>LOW STOCK</small><h2>{report["low_stock_records"]}</h2>
          </div>
        </div>
        '''

        quality = f'''
        ### Data Quality Report
        - Raw rows: **{report["raw_rows"]}**
        - Clean rows: **{report["clean_rows"]}**
        - Missing values in raw data: **{report["raw_missing"]}**
        - Duplicate orders removed: **{report["duplicates_removed"]}**
        - Missing prices filled: **{report["missing_prices_filled"]}**
        - Invalid quantity rows removed: **{report["invalid_quantity_rows_removed"]}**
        - Product source: **{report["data_source"]}**
        - API status: **{report["api_status"]}**
        '''

        insights = "\n".join(
            f"- {item}"
            for item in generate_insights(data, summary, report["data_source"])
        )

        return (
            kpis,
            data,
            summary,
            get_top_products(data),
            get_low_stock(data),
            create_revenue_chart(summary),
            insights,
            quality,
            str(clean_file),
            str(summary_file)
        )

    except Exception as error:
        return (
            "<h3>Analysis could not be completed.</h3>",
            pd.DataFrame(),
            pd.DataFrame(),
            pd.DataFrame(),
            pd.DataFrame(),
            None,
            "",
            f"Please verify the uploaded CSV. Error: {error}",
            None,
            None
        )

## 17. Assemble the Gradio Dashboard

The following cell connects the reusable pipeline to the UI components.

The application is intentionally divided into tabs so a user can move from
high-level KPIs to detailed data and operational reports.

In [ ]:
with gr.Blocks(
    title="SmartRetail Insight Studio",
    theme=gr.themes.Soft()
) as app:

    gr.Markdown(
        "# SmartRetail Insight Studio\n"
        "### Sales Intelligence • Data Quality • API Enrichment\n\n"
        "Upload a sales CSV and run the complete analytical workflow."
    )

    with gr.Row():
        upload = gr.File(
            label="Upload Sales CSV",
            file_types=[".csv"],
            type="filepath"
        )
        run_button = gr.Button("Run Analysis", variant="primary")

    kpi_output = gr.HTML()

    with gr.Tabs():

        with gr.Tab("Dashboard"):
            with gr.Row():
                chart_output = gr.Plot(label="Revenue by Category")
                insight_output = gr.Markdown(label="Insights")

        with gr.Tab("Cleaned Data"):
            clean_output = gr.Dataframe(interactive=False)

        with gr.Tab("Category Summary"):
            summary_output = gr.Dataframe(interactive=False)

        with gr.Tab("Top Products"):
            top_output = gr.Dataframe(interactive=False)

        with gr.Tab("Low Stock"):
            low_output = gr.Dataframe(interactive=False)

        with gr.Tab("Data Quality"):
            quality_output = gr.Markdown()

    with gr.Row():
        clean_download = gr.File(label="Final Sales Report")
        summary_download = gr.File(label="Category Summary")

    run_button.click(
        analyze_for_ui,
        inputs=upload,
        outputs=[
            kpi_output,
            clean_output,
            summary_output,
            top_output,
            low_output,
            chart_output,
            insight_output,
            quality_output,
            clean_download,
            summary_download
        ]
    )

print("Dashboard assembled successfully.")

## 18. Launch the Final Web Application

This is the final execution cell.

Running it starts the browser-based Gradio interface. The notebook therefore
serves two purposes:

1. **Technical documentation and development**
2. **Interactive end-user application**

The same architecture can later be moved into a dedicated application file for
deployment without changing the core analysis functions.

In [ ]:
app.launch()

# Final Project Architecture

```text
SmartRetail Insight Studio
│
├── Input
│   └── sales_data.csv
│
├── Data Quality
│   ├── missing-value detection
│   ├── duplicate detection
│   └── invalid quantity detection
│
├── Cleaning
│   ├── duplicate removal
│   ├── category standardisation
│   ├── median-price filling
│   └── invalid-row removal
│
├── API Integration
│   ├── GET request
│   ├── JSON parsing
│   └── fallback CSV
│
├── Enrichment
│   └── product_id merge
│
├── Analytics
│   ├── revenue
│   ├── stock status
│   ├── high-value classification
│   ├── category summary
│   └── top/low-stock analysis
│
├── Outputs
│   ├── CSV reports
│   └── revenue chart
│
└── Presentation
    └── Gradio web dashboard
```

## Recommended submission title

**SmartRetail Insight Studio: Interactive Sales Intelligence and API-Enriched Data Quality Analyzer**

## Resume-ready description

**Developed an interactive Python retail analytics platform that cleans transactional
data, integrates public API product information with fallback handling, computes
business KPIs and generates downloadable reports through a browser-based Gradio dashboard.**

## Key technical skills demonstrated

**Python • Pandas • Data Cleaning • CSV • REST API • JSON • Requests • Data Merging •
Business Analytics • Matplotlib • Modular Programming • Error Handling • Gradio • UI Design**